# Listed Transactions Analysis
## Step 1: Imported Libraries

In [8]:
import pandas as pd
import glob
import os

## Step 2: Loaded Listed CSV Files

In [9]:
listed_files = glob.glob("../data/CRMLSListing*.csv")
print(len(listed_files))
print(listed_files)

27
['../data\\CRMLSListing202401.csv', '../data\\CRMLSListing202402.csv', '../data\\CRMLSListing202403.csv', '../data\\CRMLSListing202404.csv', '../data\\CRMLSListing202405.csv', '../data\\CRMLSListing202406.csv', '../data\\CRMLSListing202407.csv', '../data\\CRMLSListing202408.csv', '../data\\CRMLSListing202409.csv', '../data\\CRMLSListing202410.csv', '../data\\CRMLSListing202411.csv', '../data\\CRMLSListing202412.csv', '../data\\CRMLSListing202501.csv', '../data\\CRMLSListing202502.csv', '../data\\CRMLSListing202503.csv', '../data\\CRMLSListing202504.csv', '../data\\CRMLSListing202505.csv', '../data\\CRMLSListing202506.csv', '../data\\CRMLSListing202507.csv', '../data\\CRMLSListing202508.csv', '../data\\CRMLSListing202509.csv', '../data\\CRMLSListing202510.csv', '../data\\CRMLSListing202511.csv', '../data\\CRMLSListing202512.csv', '../data\\CRMLSListing202601.csv', '../data\\CRMLSListing202602.csv', '../data\\CRMLSListing202603.csv']


## Step 3: Explored a Single File
Inspected one file before combining — checked shape, columns, data types, and sample rows.

In [10]:
df_sample = pd.read_csv(listed_files[0], low_memory=False, encoding="latin-1")
print(df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample.dtypes)
print(df_sample.head(3))

(27454, 84)
['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus', 'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1', 'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1', 'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'Cont

## Step 4: Combined All 27 Listed Files
Stacked all monthly CSVs (Jan 2024 - Mar 2026) into one DataFrame.

In [11]:
dfs = [pd.read_csv(f, low_memory=False, encoding="latin-1") for f in listed_files]
df_listed = pd.concat(dfs, ignore_index=True)
print(df_listed.shape)
print(df_listed.columns.tolist())

(853413, 84)
['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus', 'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1', 'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1', 'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'Con

## Step 5: Explored the Combined Dataset
Checked null counts and unique values in key categorical fields.

In [12]:
print(df_listed.isnull().sum().sort_values(ascending=False))
print(df_listed["PropertyType"].unique())

ElementarySchoolDistrict        853413
CoveredSpaces                   853413
AboveGradeFinishedArea          853413
FireplacesTotal                 853413
MiddleOrJuniorSchoolDistrict    853413
                                 ...  
PropertyType.1                       0
ListingKeyNumeric                    0
DaysOnMarket.1                       0
MlsStatus                            0
ListingContractDate                  0
Length: 84, dtype: int64
<StringArray>
[ 'ManufacturedInPark',      'CommercialSale',         'Residential',
    'ResidentialLease',                'Land',   'ResidentialIncome',
     'CommercialLease', 'BusinessOpportunity']
Length: 8, dtype: str


In [13]:
critical_cols = ["ListPrice","OriginalListPrice","LivingArea","DaysOnMarket","PropertyType","ListingContractDate","City","PostalCode","BedroomsTotal","BathroomsTotalInteger","MlsStatus"]
print(df_listed[critical_cols].isnull().sum())

ListPrice                  2134
OriginalListPrice          3313
LivingArea               106455
DaysOnMarket                  0
PropertyType                  0
ListingContractDate           0
City                        961
PostalCode                  208
BedroomsTotal            103069
BathroomsTotalInteger     69953
MlsStatus                     0
dtype: int64


## Step 6: Cleaned the Data
- Filtered to Residential only
- Dropped rows missing ListPrice or LivingArea
- Dropped duplicate .1 columns from API script
- Converted date columns to datetime
- Dropped columns that were mostly null or irrelevant

In [14]:
df_listed = df_listed[df_listed["PropertyType"] == "Residential"]
df_listed = df_listed.dropna(subset=["ListPrice", "LivingArea"])
print(df_listed.shape)

(539867, 84)


In [15]:
df_listed["ListingContractDate"] = pd.to_datetime(df_listed["ListingContractDate"])
df_listed["ContractStatusChangeDate"] = pd.to_datetime(df_listed["ContractStatusChangeDate"])
df_listed["PurchaseContractDate"] = pd.to_datetime(df_listed["PurchaseContractDate"])
df_listed["CloseDate"] = pd.to_datetime(df_listed["CloseDate"])
print(df_listed[["ListingContractDate","ContractStatusChangeDate","PurchaseContractDate","CloseDate"]].dtypes)

ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
PurchaseContractDate        datetime64[us]
CloseDate                   datetime64[us]
dtype: object


In [16]:
cols_to_drop = ["PropertyType.1","ListAgentFirstName.1","DaysOnMarket.1","LivingArea.1","Longitude.1","Latitude.1","ListPrice.1","ListAgentLastName.1","CloseDate.1","BuyerOfficeName.1","UnparsedAddress.1","BuyerAgencyCompensationType","BuyerAgencyCompensation","BusinessType","TaxYear","AboveGradeFinishedArea","MiddleOrJuniorSchoolDistrict"]
cols_to_drop = [c for c in cols_to_drop if c in df_listed.columns]
df_listed = df_listed.drop(columns=cols_to_drop)
print(df_listed.shape)

(539867, 67)


In [17]:
df_listed.isnull().sum().sort_values(ascending=False)

CoveredSpaces               539867
FireplacesTotal             539867
TaxAnnualAmount             539867
ElementarySchoolDistrict    539867
BelowGradeFinishedArea      536841
                             ...  
PropertyType                     0
ListOfficeName                   0
DaysOnMarket                     0
ListingContractDate              0
ListingId                        0
Length: 67, dtype: int64

## Step 7: Removed Outliers Using IQR
Applied IQR method to ListPrice, LivingArea, and DaysOnMarket.

In [18]:
for col in ["ListPrice", "LivingArea", "DaysOnMarket"]:
    Q1 = df_listed[col].quantile(0.25)
    Q3 = df_listed[col].quantile(0.75)
    IQR = Q3 - Q1
    df_listed = df_listed[(df_listed[col] >= Q1 - 1.5*IQR) & (df_listed[col] <= Q3 + 1.5*IQR)]
print(df_listed.shape)

(437770, 67)


## Step 8: Feature Engineering
Created new calculated columns: ListPricePerSqFt, PriceRatio, ListYear, ListMonth, YrMo.

In [19]:
df_listed["ListPricePerSqFt"] = df_listed["ListPrice"] / df_listed["LivingArea"]
df_listed["PriceRatio"] = df_listed["ClosePrice"] / df_listed["ListPrice"]
df_listed["ListYear"] = df_listed["ListingContractDate"].dt.year
df_listed["ListMonth"] = df_listed["ListingContractDate"].dt.month
df_listed["YrMo"] = df_listed["ListingContractDate"].dt.to_period("M").astype(str)
print(df_listed[["ListPrice","LivingArea","ListPricePerSqFt","PriceRatio","ListYear","ListMonth","YrMo"]].head(3))

   ListPrice  LivingArea  ListPricePerSqFt  PriceRatio  ListYear  ListMonth  \
3  2500000.0      2788.0        896.700143         NaN      2024          1   
7   665000.0      1487.0        447.209146         NaN      2024          1   
8  1125000.0      1750.0        642.857143         NaN      2024          1   

      YrMo  
3  2024-01  
7  2024-01  
8  2024-01  


## Step 9: Exported Clean CSV

In [20]:
df_listed.to_csv("../data/listed_clean.csv", index=False)
print("Exported:", df_listed.shape)

Exported: (437770, 72)
